# Cluster Validation

## Overview

Clustering produces a result regardless of whether any real cluster structure exists. Validation answers two questions:
1. **Is there cluster structure at all?**
2. **How good is this clustering?**

| Index | Range | Better | Needs ground truth? |
|---|---|---|---|
| Silhouette | -1 to 1 | Higher | No |
| Calinski-Harabasz | 0 to inf | Higher | No |
| Davies-Bouldin | 0 to inf | Lower | No |
| Adjusted Rand Index | -1 to 1 | Higher | Yes |
| Normalised Mutual Info | 0 to 1 | Higher | Yes |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (silhouette_score, silhouette_samples,
    calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score, normalized_mutual_info_score)

rng = np.random.default_rng(42)
n = 300
clusters_true = rng.choice([0,1,2], n)
means = np.array([[50,8,7.8],[200,4,7.2],[350,1,6.6]])
X_raw = np.vstack([rng.normal(means[c],[30,1,0.3],(1,3)) for c in clusters_true])
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)
print(f"Data: {X.shape}, true clusters: {np.bincount(clusters_true)}")

---
## Internal Validation across k

In [ ]:
ks = range(2, 9)
metrics = {"k":[],"silhouette":[],"calinski_harabasz":[],"davies_bouldin":[]}
for k in ks:
    labels = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(X)
    metrics["k"].append(k)
    metrics["silhouette"].append(silhouette_score(X, labels))
    metrics["calinski_harabasz"].append(calinski_harabasz_score(X, labels))
    metrics["davies_bouldin"].append(davies_bouldin_score(X, labels))
df_m = pd.DataFrame(metrics).set_index("k")
print(df_m.round(3))
fig, axes = plt.subplots(1,3,figsize=(13,4))
for ax, col, better in zip(axes,
    ["silhouette","calinski_harabasz","davies_bouldin"],
    ["higher","higher","lower"]):
    ax.plot(ks, df_m[col], "o-", color="steelblue", lw=2)
    best_k = df_m[col].idxmax() if better=="higher" else df_m[col].idxmin()
    ax.axvline(best_k, color="#e74c3c", lw=1.5, linestyle="--", label=f"Best k={best_k}")
    ax.set_xlabel("k"); ax.set_title(f"{col} ({better} is better)"); ax.legend()
plt.tight_layout(); plt.show()

---
## External Validation: ARI and NMI

In [ ]:
print("External validation (requires ground truth labels)")
for k in [2, 3, 4]:
    labels = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(X)
    ari = adjusted_rand_score(clusters_true, labels)
    nmi = normalized_mutual_info_score(clusters_true, labels)
    print(f"k={k}: ARI={ari:.3f}, NMI={nmi:.3f}")
print("\nARI=1.0 perfect; ARI~0 chance; ARI can be negative")
print("NMI=1.0 perfect; 0.0 no shared information")

---
## Silhouette Plot

In [ ]:
km_best = KMeans(n_clusters=3, n_init=50, random_state=42)
labels_best = km_best.fit_predict(X)
sil_vals = silhouette_samples(X, labels_best)
mean_sil = silhouette_score(X, labels_best)
fig, ax = plt.subplots(figsize=(8,5))
y_pos = 0
colors = ["#e74c3c","steelblue","#4fffb0"]
for cl, color in enumerate(colors):
    vals = np.sort(sil_vals[labels_best==cl])
    ax.barh(range(y_pos, y_pos+len(vals)), vals, height=1, color=color, alpha=0.8,
            label=f"Cluster {cl} (n={len(vals)})")
    y_pos += len(vals) + 8
ax.axvline(mean_sil, color="navy", lw=2, linestyle="--", label=f"Mean={mean_sil:.3f}")
ax.set_xlabel("Silhouette coefficient")
ax.set_title("Silhouette Plot -- wide positive bars indicate well-separated clusters")
ax.legend(); plt.tight_layout(); plt.show()
print(f"Negative silhouette observations: {(sil_vals<0).sum()} ({(sil_vals<0).mean():.1%})")

---
## Comparing Algorithms

In [ ]:
km = KMeans(n_clusters=3, n_init=20, random_state=42).fit_predict(X)
gm = GaussianMixture(n_components=3, n_init=10, random_state=42).fit(X).predict(X)
db = DBSCAN(eps=0.5, min_samples=5).fit_predict(X)
header = f"{'Algorithm':20s} {'Silhouette':>12} {'CH':>12} {'DB':>10} {'ARI':>10}"
print(header)
for name, labels in [('K-Means',km),('GMM',gm),('DBSCAN',db)]:
    mask = labels != -1
    if mask.sum() < 2 or len(set(labels[mask])) < 2:
        print(f'{name:20s}  insufficient clusters'); continue
    sil = silhouette_score(X[mask], labels[mask])
    ch  = calinski_harabasz_score(X[mask], labels[mask])
    db_ = davies_bouldin_score(X[mask], labels[mask])
    ari = adjusted_rand_score(clusters_true[mask], labels[mask])
    row = f'{name:20s} {sil:12.3f} {ch:12.1f} {db_:10.3f} {ari:10.3f}'
    print(row)

---

## Common Pitfalls

**1. Optimising a single internal index**  
Different indices favour different geometries. Silhouette favours compact clusters; Davies-Bouldin penalises overlap; Calinski-Harabasz grows with n. Report all three and look for consensus alongside domain knowledge.

**2. Using unadjusted Rand Index**  
Raw Rand Index is inflated by chance for large k. Always use Adjusted Rand Index (ARI), which has expected value 0 for random assignments.

**3. Including DBSCAN noise points in internal indices**  
Silhouette, CH, and DB require every point to be assigned to a cluster. Always filter to `labels != -1` before computing any internal index for DBSCAN.

**4. Selecting k by indices alone without profiling clusters**  
High silhouette does not guarantee meaningful clusters. Always profile centroids and distributions — statistical validity and substantive interpretability are both required.

**5. Not testing for cluster tendency before clustering**  
If data have no cluster structure, any algorithm will produce clusters anyway. Use the Hopkins statistic to test whether genuine cluster structure exists before interpreting results.
---
*python_methods_library - Samantha McGarrigle*